**Sam Polyakov and Teagan Turner**

Fall 2025

CS 343: Neural Networks

In [13]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

plt.show()
plt.style.use(['seaborn-v0_8-colorblind', 'seaborn-v0_8-darkgrid'])
plt.rcParams.update({'font.size': 16})

np.set_printoptions(suppress=True, precision=3)

# Automatically reload external modules
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## TensorFlow install test

*Sanity check that TensorFlow is installed correctly:*

Executing the following cell should print 3 

In [14]:
tf.print(tf.reduce_sum([tf.constant(1), tf.constant(2)]))

3


# Project 4 | Transfer Learning

## Task 1: Implement ConvNet4AccelV2 in TensorFlow

Construct the familiar `ConvNet4AccelV2` neural network architecture from last project in TensorFlow's high level `Keras::Sequential` API. Also like your last project, train on the STL-10 training set and test on the STL-10 test set.

### 1a. Use the high level `Keras::Sequential` API in TensorFlow to implement the architecture of ConvNet4AccelV2 from the last project. Train and test your network on the STL-10 dataset. 

**Goal:** Achieve ≥ 47% on either the validation set or test set. *For our purposes, getting ≥ 47% validation accuracy at any point during training is enough (i.e. doesn't need to be at the very end of training).*

#### Notes

- You should use the usual STL-10 data acquisition and preprocessing code from your last project. You can use the default split, or modify it yourself.
- You don't need to do a hyperparameter search. Values that worked on the CNN project should get you in the ballpark here. The goal is to show that you know how to put together a `keras::Sequential` model and have it work successfully.
- To achieve the val/test target accuracy, you may have to tweak the hyperparameters by hand a little (for example: learning rate, number of epochs, regularization, number of hidden units, ...), but it should not take too much effort.
- TensorFlow needs the RGB color channel AFTER the spatial dimensions. For example: (32, 32, 3), not (3, 32, 32). You may therefore need to slightly modify the preprocesssing pipeline for this project.

#### Keras Sequential workflow

Recall the `Keras::Sequential` common workflow:

- Build structure of network with `Keras::Sequential`.
- Compile network with your choice of optimizer, loss, and metrics.
- Fit the model (remembering to pass in the appropriate training and validation sets). This results a history object that can be used to examine training/validation accuracy and loss.
- Evaluate the model on the test set. This returns test loss and accuracy.

#### Helpful documentation

These documentation pages should be helpful:
- https://www.tensorflow.org/api_docs/python/tf/keras/Sequential
- https://www.tensorflow.org/api_docs/python/tf/keras/Model#compile
- https://www.tensorflow.org/api_docs/python/tf/keras/Model#evaluate
- https://www.tensorflow.org/api_docs/python/tf/keras/Model#fit
- https://www.tensorflow.org/api_docs/python/tf/keras/Model#summary
- https://www.tensorflow.org/api_docs/python/tf/keras/optimizers

In [15]:
import load_stl10_dataset
from preprocess_data import load_stl10

In [25]:
classes = np.loadtxt(os.path.join('data', 'stl10_binary', 'class_names.txt'), dtype=str)

load_stl10_dataset.purge_cached_dataset()

# YOUR CODE HERE

x_train, y_train, x_test, y_test, x_val, y_val, x_dev, y_dev = load_stl10(
    n_train_samps=4000, n_test_samps=500, n_valid_samps=499, n_dev_samps=1, scale_fact=3)



# Convert to float and normalize
x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32") / 255.0
x_val   = x_val.astype("float32") / 255.0
x_dev   = x_dev.astype("float32") / 255.0

# Convert channels-first (N, 3, H, W) → channels-last (N, H, W, 3)
x_train = x_train.transpose(0, 2, 3, 1)
x_test  = x_test.transpose(0, 2, 3, 1)
x_val   = x_val.transpose(0, 2, 3, 1)
x_dev   = x_dev.transpose(0, 2, 3, 1)

# Ensure labels are integers
y_train = y_train.astype("int64")
y_test  = y_test.astype("int64")
y_val   = y_val.astype("int64")
y_dev   = y_dev.astype("int64")

print("Train:", x_train.shape, y_train.shape)
print("Val:",   x_val.shape,   y_val.shape)
print("Test:",  x_test.shape,  y_test.shape)
print("Dev:",   x_dev.shape,   y_dev.shape)



Images are: (5000, 96, 96, 3)
Labels are: (5000,)
Resizing 5000 images to 32x32...Done!
Saving Numpy arrays the images and labels to ./numpy...Done!
Train: (4000, 32, 32, 3) (4000,)
Val: (499, 32, 32, 3) (499,)
Test: (500, 32, 32, 3) (500,)
Dev: (1, 32, 32, 3) (1,)


In [32]:
tf.random.set_seed(0)

# YOUR CODE HERE
# Train (same hyperparams as last project)

wt_scale = 1e-2
reg = 0.0
dropout_rate = 0.5  

initializer = tf.keras.initializers.RandomNormal()
l2 = None

model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, kernel_size=7, padding='same', activation='relu',
                           kernel_initializer=initializer, bias_initializer='zeros',
                           kernel_regularizer=l2,
                           input_shape=(32, 32, 3), name='Conv2D0'),
    tf.keras.layers.MaxPooling2D(pool_size=2, strides=2, name='MaxPool2D1'),
    tf.keras.layers.Flatten(name='Flatten2'),
    tf.keras.layers.Dense(100, activation='relu',
                          kernel_initializer=initializer, bias_initializer='zeros',
                          kernel_regularizer=l2,
                          name='Dense3'),
    tf.keras.layers.Dropout(dropout_rate, name='Dropout4'),
    tf.keras.layers.Dense(10, activation='softmax',
                          kernel_initializer=initializer, bias_initializer='zeros',
                          kernel_regularizer=l2,
                          name='Dense5'),
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

batch_size = 128
epochs = 35

history = model.fit(
    x_train, y_train,
    batch_size=batch_size,
    epochs=epochs,
    validation_data=(x_val, y_val),
    verbose=2
)


Epoch 1/35
32/32 - 2s - 47ms/step - accuracy: 0.1472 - loss: 2.2532 - val_accuracy: 0.2465 - val_loss: 2.1383
Epoch 2/35
32/32 - 1s - 20ms/step - accuracy: 0.2545 - loss: 2.0348 - val_accuracy: 0.2665 - val_loss: 1.9738
Epoch 3/35
32/32 - 1s - 20ms/step - accuracy: 0.2950 - loss: 1.9107 - val_accuracy: 0.2906 - val_loss: 1.8858
Epoch 4/35
32/32 - 1s - 22ms/step - accuracy: 0.3120 - loss: 1.8496 - val_accuracy: 0.2946 - val_loss: 1.8355
Epoch 5/35
32/32 - 1s - 22ms/step - accuracy: 0.3288 - loss: 1.8054 - val_accuracy: 0.3206 - val_loss: 1.7868
Epoch 6/35
32/32 - 1s - 20ms/step - accuracy: 0.3420 - loss: 1.7667 - val_accuracy: 0.3166 - val_loss: 1.7480
Epoch 7/35
32/32 - 1s - 20ms/step - accuracy: 0.3590 - loss: 1.7298 - val_accuracy: 0.3327 - val_loss: 1.7185
Epoch 8/35
32/32 - 1s - 22ms/step - accuracy: 0.3750 - loss: 1.6929 - val_accuracy: 0.3427 - val_loss: 1.6911
Epoch 9/35
32/32 - 1s - 20ms/step - accuracy: 0.3895 - loss: 1.6607 - val_accuracy: 0.3467 - val_loss: 1.6665
Epoch 10/3

### 1b. Make 2 "high quality" plots showing the following

- Training and validation accuracy (y axis) over training epochs (x axis).
- Training and validation loss (y axis) over epochs (x axis).

A high quality plot consists of:
- A useful title
- X and Y axis labels
- A legend

In [ ]:
# YOUR CODE HERE

### 1c. Visualize predictions

Make a 5x5 grid of the first 25 images in the test dataset. Label each with the predicted class label string (English label, not an int code).

*NOTE: You standardized your images during preprocessing, so, for example, some pixel features are now negative. To make the images look like they should, either write code to grab an un-preprocessed version of the test images or min-max normalize the images prior to plotting them.*

In [ ]:
# YOUR CODE HERE

### 1d. Questions

**Question 1:** What accuracy do you get on the STL-10 test set? Briefly summarize any non-default hyperparameters that you used to obtain this result.

**Question 2:** How do the loss and accurary results compare to the CNN project?

**Question 3:** Identify a few misclassifications. Is there a discernable pattern?

**Answer 1:**

YOUR ANSWER HERE

**Answer 2:**

YOUR ANSWER HERE

**Answer 3:**

YOUR ANSWER HERE

## Task 2: Transfer learning

Here you will use TensorFlow to download the pre-trained MobileNetV2 network (you may also use another network like InceptionV3, VGG19, or EfficientNet, but MobileNetV2 likely will run noticeably faster on your machine). We will use transfer learning to accelerate training to solve a novel problem: **the binary classification task of discriminating whether an image is of a hotdog or not.**

### Overview

- Remove the output layer, add a new Dense output layer.
- Freeze (disable) training on all non-output layers.
- Train the last layer on a food dataset. Assess performance. Plot some example images and their classification below.

### 2a. Download and load in hotdot image dataset

Download the **food dataset** from the project website, copy it into a `data` subfolder in your project directory.

Run the below code to load in the hot-dog-or-not dataset. Check the shapes to ensure everything is loaded in correctly. 

In [ ]:
ds_base_dir='data/hot-dog-not-hot-dog/numpy/'

hotdog_train_x = np.load(os.path.join(ds_base_dir, 'train_x.npy'))
hotdog_train_y = np.load(os.path.join(ds_base_dir, 'train_y.npy'))
hotdog_test_x = np.load(os.path.join(ds_base_dir, 'test_x.npy'))
hotdog_test_y = np.load(os.path.join(ds_base_dir, 'test_y.npy'))

print(f'Training hotdog split shape: {hotdog_train_x.shape}. Should be (16000, 96, 96, 3)')
print(f'Test hotdog split shape: {hotdog_test_x.shape}. Should be (4000, 96, 96, 3)')

### 2b. Normalize hotdog dataset

Standardize both the train and test dataset according to the **same statistics** computed from the **training set**.

In [ ]:
# YOUR CODE HERE

### 2c. Create hotdog validation set

Set aside the last 20% of the training set as the validation set

In [ ]:
# YOUR CODE HERE

In [ ]:
print(f'Validation hotdog split shape: {hotdog_val_x.shape}. Should be (3200, 96, 96, 3)')
print(f'Training hotdog split shape: {hotdog_train_x.shape}. Should be (12800, 96, 96, 3)')

### 2d. Load in pre-trained MobileNetV2 network.

Load in a pre-trained MobileNetV2 network (look up constructor in [tf.keras.applications](https://www.tensorflow.org/api_docs/python/tf/keras/applications/mobilenet_v2/MobileNetV2) or look at the tutorial from class) and set it to a variable called `model`. Remember to make the network not trainable. Calling the `summary()` method on the network object should show you a table with many rows. The top and bottom rows should be:

    Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
    ┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
    │ input_layer_19      │ (None, 96, 96, 3) │          0 │ -                 │
    │ (InputLayer)        │                   │            │            
                                        
    ├─────────────────────┼───────────────────┼────────────┼───────────────────┤
    │ out_relu (ReLU)     │ (None, 3, 3,      │          0 │ Conv_1_bn[0][0]   │
    │                     │ 1280)             │            │                   │  
                                                                                                  
==================================================================================================

and you should see the following at the bottom:

    Total params: 2,257,984 (8.61 MB)
    Trainable params: 0 (0.00 B)
    Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
# YOUR CODE HERE

### 2e. Create augmented model

Create a new `keras::Sequential` augmented model with an output layer that has the correct number of units to deal with the hot-dog or not problem with your choice of optimizer, an appropriate loss, and metric(s).

#### Helpful links

https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense

https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam

https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/AdamW

https://www.tensorflow.org/api_docs/python/tf/losses

In [ ]:
# YOUR CODE HERE

### 2f. Questions

**Question 4:** What is the accuracy and loss for the network with the untrained output layer on the test set? Explain why you got the accuracy value that you did.

**Question 5:** Briefly defend your choice of number of units in the output layer.

**Answer 4:**

YOUR ANSWER HERE

**Answer 5:**

YOUR ANSWER HERE

In [ ]:
# YOUR CODE HERE

### 2g. Fit the augmented model on the hotdog training data

Setting the verbose optional parameter to 2 will give you helpful printouts of performance on the validation set as it completes every epoch of training.

#### Training goal

The aim is to achieve at least 85% accuracy on the validation set. If everything is set up properly, you should only need to train for a very small number of epochs.


#### Note

If training time is taking much more than a few minutes per epoch on your computer, you could try reducing the number of data samples in train and validation. For example, by default train `N = 12800`. Try `N = 6400` instead. You could do the same for the validation set.

In [ ]:
# YOUR CODE HERE

### 2h. Plot hotdog results

Produce 2 high quality plots showing the following:

- Training and validation loss over epoch.
- Training and validation accuracy over epoch.

In [ ]:
# YOUR CODE HERE

### 2i. Visualize predictions on test set

Use your trained hotdog classifier to get the predicted classes for the first 25 **test set** samples. Then create a 5x5 grid of the first 25 test samples and label each with the predicted class string (English label, not an int code).
- If the prediction is correct, color the label *blue*.
- If the prediction is incorrect, color the label *red*.

**Note:**
- Depending on how you get the network predictions, TensorFlow may give you a vector of class probabilities. This could be shape `(N, 2)` or `(N,)`, depending on the output layer activation function that you used. Remember that if you have these probabilities, you need to recover the predicted class index (e.g. `0`, `1`) before you can label your plots.
- If `imshow` gives you warnings about clipping, check the range of the test samples. If your max is slightly higher than 1, either ignore the warning or divide by the max. 

In [ ]:
# YOUR CODE HERE